# Phase 1 — data loaders walkthrough

Demonstrates the three loaders in `zlabel.data` on real downloaded files. No labeling logic yet — that starts in Phase 2 (panels) and Phase 3 (grounding).

The three loaders form one pipeline: **ZFA** gives an anatomy graph, **ZFIN** gives expression evidence, and the **GAF** resolves gene synonyms — the three inputs Phase 2 consumes.

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads the ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
    <br><br>
    Alternatively, run the two commented cells below to set the data up from inside the notebook.
    
</div>

In [13]:
import zlabel; print(f"zlabel {zlabel.__version__}")
from pathlib import Path
import os, sys

# Relative to CWD — correct when the server is started from the repo root.
PACKAGE ="zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / Path("data/ontologies")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )
print(f"data at {DATA_DIR.resolve()}")

zlabel 0.1.0
data at /home/ec2-user/PycharmProjects/zlabel/data/ontologies


## 1. ZFA — Zebrafish Anatomy Ontology

The ZFA is a directed graph of anatomical terms: a **term** is any named structure (e.g. `ZFA:0000051` — *aorta*), and a **directed edge** points from a more specific term to a broader one, labelled by the relationship that connects them.

```
 ─────────────────────────────────────────────────────────────────────────────────────────────────
|  aorta  ───is_a───▶  blood vessel  ───is_a───▶  vessel  ───part_of───▶  cardiovascular system   |
 ─────────────────────────────────────────────────────────────────────────────────────────────────
```

`load_zfa` loads this into a `networkx.MultiDiGraph` — nodes are ZFA term IDs, edges go child → parent keyed by relationship type (`is_a`, `part_of`, `develops_from`).

`ancestors` does a BFS walk up that graph, which gives you **all transitive ancestors in nearest-first order** — not just immediate parents, but everything reachable by following the chosen edge types. Two edge types give two different views of the same structure:

- **`is_a + part_of` (default)** — structural lineage: what category of thing is this, and what larger structure does it belong to? Aorta → blood vessel → cardiovascular system.
- **`develops_from`** — developmental lineage: what embryonic precursor did this arise from? Aorta → lateral plate mesoderm → mesoderm. Useful when the biology question is about cell fate, not mature anatomy.

The reason this matters for the labeller: a gene annotated as expressing *"in aorta"* implicitly expresses in every ancestor too. Walking the graph propagates that evidence upward to broader anatomical contexts.

In [24]:
zfa = zlabel.load_zfa(DATA_DIR / "zfa.obo")
print(f"ZFA: {len(zfa.nodes):,} terms, {len(zfa.edges):,} edges\n")

# Find all ZFA terms whose name contains "aorta" — the walrus operator assigns
# term_name() while filtering so it's only called once per node.
aorta_hits = sorted(
    (
        (nid, name)
        for nid in zfa.nodes
        if (name := zlabel.term_name(zfa, nid)) and "aorta" in name
    ),
    key=lambda x: x[1],
)
print("Terms containing 'aorta':")
for tid, name in aorta_hits:
    print(f"  {tid}  {name}")
print()

# aorta_hits is sorted alphabetically, so [0] is "aorta" itself rather than
# a sub-structure like "aortic arch". Walking its ancestors shows the full
# structural context that any aorta-annotated gene inherits.
tid, name = aorta_hits[0]
parents = zlabel.ancestors(zfa, tid)
print(f"ancestors of {name!r} ({tid}) — BFS, is_a + part_of (default):")
for pid in parents:
    print(f"  {pid}  {zlabel.term_name(zfa, pid)}")

ZFA: 3,161 terms, 12,293 edges

Terms containing 'aorta':
  ZFA:0000014  dorsal aorta
  ZFA:0001054  lateral dorsal aorta
  ZFA:0000604  ventral aorta
  ZFA:0005028  ventral wall of dorsal aorta

ancestors of 'dorsal aorta' (ZFA:0000014) — BFS, is_a + part_of (default):
  ZFA:0000005  artery
  ZFA:0005024  trunk vasculature
  ZFA:0005295  axial blood vessel
  ZFA:0005314  blood vessel
  ZFA:0001079  blood vasculature
  ZFA:0005249  vasculature
  ZFA:0001115  trunk
  ZFA:0001073  axial vasculature
  ZFA:0001488  multi-tissue structure
  ZFA:0001490  cavitated compound organ
  ZFA:0000010  cardiovascular system
  ZFA:0001308  organism subdivision
  ZFA:0000037  anatomical structure
  ZFA:0000496  compound organ
  ZFA:0001439  anatomical system
  ZFA:0001094  whole organism
  ZFA:0100000  zebrafish anatomical entity
  ZFA:0001512  anatomical group


## 2. ZFIN wildtype expression

`load_zfin_expression` parses the 15-column ZFIN TSV into a dict keyed by lowercased gene symbol. Each value is a list of `ZfinExpressionRecord` objects — one per curated observation — carrying the most specific ZFA structure the gene was seen in plus the ZFS stage window.

These are the **in-vivo grounding signals**: where do a cluster's markers actually express?

<div class="alert alert-info" role="alert">
    <b>ℹ️ Why this matters</b> — combined with the ZFA ancestor walk from Section 1, an observation in a specific structure (e.g. <code>dorsal aorta</code>) also counts as evidence for every broader structure above it. Phase 3 uses this to score how well a marker set matches a candidate tissue.
    
</div>

In [27]:
from collections import Counter


expr = zlabel.load_zfin_expression(DATA_DIR / "zfin_wildtype_expression.txt")
print(f"ZFIN expression: {len(expr):,} genes with curated records\n")

# Three canonical markers spanning different broad lineages.
for gene in ("kdrl", "gata1a", "sox17"):
    records = expr.get(gene, [])
    structure_counts = Counter((r.zfa_id, r.zfa_name) for r in records)

    print(f"{gene}:  {len(records)} records, {len(structure_counts)} unique structures")
    for (zfa_id, zfa_name), n in sorted(structure_counts.items(), key=lambda x: x[0][1])[:6]:
        print(f"    {zfa_id}  {zfa_name}  ({n})")
    if len(structure_counts) > 6:
        print(f"    ... {len(structure_counts) - 6} more")
    print()

ZFIN expression: 14,485 genes with curated records

kdrl:  482 records, 75 unique structures
    ZFA:0009258  angioblastic mesenchymal cell  (8)
    ZFA:0005604  angiogenic sprout  (5)
    ZFA:0005039  anterior lateral mesoderm  (3)
    ZFA:0005041  anterior lateral plate mesoderm  (5)
    ZFA:0005004  aortic arch  (2)
    ZFA:0000005  artery  (2)
    ... 69 more

gata1a:  529 records, 50 unique structures
    ZFA:0005039  anterior lateral mesoderm  (3)
    ZFA:0001073  axial vasculature  (1)
    ZFA:0000006  ball  (2)
    ZFA:0000007  blood  (14)
    ZFA:0009044  blood cell  (3)
    ZFA:0000094  blood island  (13)
    ... 44 more

sox17:  184 records, 20 unique structures
    ZFA:0000001  Kupffer's vesicle  (3)
    ZFA:0005041  anterior lateral plate mesoderm  (1)
    ZFA:0001176  blastoderm  (1)
    ZFA:0001175  blastodisc  (1)
    ZFA:0000186  common cardinal vein  (1)
    BSPO:0000079  dorsal region  (1)
    ... 14 more



## 3. GAF gene synonyms

`load_gene_synonym_map` inverts the GAF's synonym column (col 11) into a case-folded
dict from any alias or previous-name to the current symbol(s) that declare it.
One previous-name can fan out to multiple current paralogs.

`genes.normalize_symbol` (Phase 2) consumes this map as the first step of the
annotation loop — so a marker submitted as `kdr` or `flk1` still resolves to `kdrl`.

In [34]:
syns = zlabel.load_gene_synonym_map(DATA_DIR / "zfin.gaf")
print(f"GAF synonym map: {len(syns):,} entries (aliases + current symbols)\n")

# Aliases that frequently appear in published datasets for well-known markers.
probes = ["kdr", "flk1", "kdrl", "vegfr2", "hbae1", "hbae1.1", "gata1", "gata1a"]
for alias in probes:
    resolved = syns.get(alias.lower(), set())
    status = ", ".join(sorted(resolved)) if resolved else "(not found)"
    print(f"  {alias:<12} -> {status}")

GAF synonym map: 66,682 entries (aliases + current symbols)

  kdr          -> kdr, kdrl
  flk1         -> kdrl
  kdrl         -> kdrl
  vegfr2       -> kdrl
  hbae1        -> hbae1.1
  hbae1.1      -> hbae1.1
  gata1        -> gata1a
  gata1a       -> gata1a


## What's next

**Phase 2** adds `genes.normalize_symbol` (uses the synonym map above) and `panels.yaml` — the curated tissue/lineage buckets and the overlap scorer that produces the first-pass ranked table.

**Phase 3** wires everything together: normalized markers → panel scores → ZFIN grounding → converging-evidence decision → `Label` packet.

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 1</b>
    <br><br>
    An anatomy graph (<b>ZFA</b>), an expression lookup (<b>ZFIN</b>), and a synonym resolver (<b>GAF</b>) — the three inputs Phase 2 consumes to produce the first ranked table.
    
</div>

<br>

***FOR FUN VISUALIZE THE GRAPH***

In [ ]:
from IPython.display import HTML, display
from pyvis.network import Network

def show_subgraph_interactive(
    graph: nx.MultiDiGraph,
    term_id: str | None = None,
    output: str = "zfa.html",
) -> None:
    if term_id:
        ancestor_ids = zlabel.ancestors(graph, term_id)
        subg = graph.subgraph([term_id, *ancestor_ids])
    else:
        subg = graph

    net = Network(
        height="800px", width="100%",
        directed=True,
        bgcolor="#1e1e2e",
        font_color="#cdd6f4",
        notebook=True,
        cdn_resources="in_line",  # embeds vis.js directly — avoids CDN failures in Jupyter
    )

    for nid in subg.nodes:
        label  = zlabel.term_name(graph, nid) or nid
        colour = "#E84C4C" if nid == term_id else "#4C9BE8"
        net.add_node(nid, label=label, title=f"{nid}\n{label}", color=colour, size=12)

    for u, v, key in subg.edges(keys=True):
        net.add_edge(u, v, title=key, color=_EDGE_COLOURS.get(key, "#666"), width=1.5)

    net.set_options("""{
      "layout": {"hierarchical": {"enabled": true, "direction": "UD", "sortMethod": "directed"}},
      "physics": {"enabled": false}
    }""")

    display(HTML(net.generate_html()))  # render inline rather than writing a file

show_subgraph_interactive(zfa, tid)